# 2.6 Les comparaisons
**Sujet 04 Allstate · P3 · HOUNGBO Gwenaël**

Question : de combien un modèle soigné bat-il l'estimation sans modèle, et **lequel** est le meilleur ?

Règle d'une comparaison honnête : **mêmes plis, même métrique (MAE), même plancher** pour tous les candidats. Une seule chose change à la fois.

Candidats :
1. **Plancher** : la médiane (voir 2.3)
2. **Ridge + one-hot** : modèle linéaire, catégorielles en one-hot (modalités rares regroupées)
3. **HGB + ordinal** : arbres de gradient, catégorielles en entiers, perte MAE
4. **HGB + ordinal + log** : idem, mais on apprend `log(loss)` puis on revient en euros

*À lancer depuis le dossier du projet. Compte plusieurs minutes. Pour un essai rapide, mets `SOUS_ECHANTILLON = 0.2` dans la cellule de chargement.*

## Chargement

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("data")          # dossier qui contient train.csv
OUT = Path("sorties")        # dossier des résultats
OUT.mkdir(exist_ok=True)

train = pd.read_csv(DATA / "train.csv")
print(train.shape)           # attendu : (150654, 132)

cat_cols = [c for c in train.columns if c.startswith("cat")]
cont_cols = [c for c in train.columns if c.startswith("cont")]
SOUS_ECHANTILLON = None      # None = toutes les lignes ; 0.2 = 20 % des lignes pour un essai rapide
if SOUS_ECHANTILLON:
    train = train.sample(frac=SOUS_ECHANTILLON, random_state=42).reset_index(drop=True)
y = train["loss"]

## Le découpage de validation (le même pour tous)

In [ ]:
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_SPLITS = 5

# Découpage : on range les montants en 10 tranches (déciles) et chaque pli reçoit
# la même proportion de chaque tranche. Ainsi la queue des gros sinistres,
# rare, se retrouve dans tous les plis.
# >>> Si l'équipe a fixé un autre découpage en 2.1, remplace ces lignes par le sien. <<<
tranches = pd.qcut(y, q=10, labels=False, duplicates="drop")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
plis = list(skf.split(np.zeros(len(y)), tranches))      # liste de (indices train, indices validation)
print(len(plis), "plis |", len(plis[0][0]), "lignes d'entraînement et", len(plis[0][1]), "de validation par pli")

## Les candidats

In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

X = train[cat_cols + cont_cols]

def plancher_mediane():
    return DummyRegressor(strategy="median")

def ridge_onehot():
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.005), cat_cols),
        ("cont", StandardScaler(), cont_cols),
    ])
    return Pipeline([("pre", pre), ("reg", Ridge(alpha=10.0))])

def hgb_ordinal():
    pre = ColumnTransformer([
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=np.nan), cat_cols),
        ("cont", "passthrough", cont_cols),
    ])
    reg = HistGradientBoostingRegressor(loss="absolute_error", max_iter=200, random_state=SEED)
    return Pipeline([("pre", pre), ("reg", reg)])

def hgb_ordinal_log():
    # on apprend log(1 + loss), puis expm1 ramène en euros
    return TransformedTargetRegressor(regressor=hgb_ordinal(), func=np.log1p, inverse_func=np.expm1)

candidats = {
    "plancher (médiane)": plancher_mediane,
    "ridge + one-hot": ridge_onehot,
    "HGB + ordinal": hgb_ordinal,
    "HGB + ordinal + log": hgb_ordinal_log,
}
list(candidats)

## Évaluation : mêmes plis pour tous

In [ ]:
import time
from sklearn.metrics import mean_absolute_error

q90 = y.quantile(0.90)
lignes = []
for nom, fabrique in candidats.items():
    t0 = time.time()
    oof = np.zeros(len(y))                                  # prédictions « hors pli »
    maes = []
    for itr, iva in plis:
        m = fabrique()
        m.fit(X.iloc[itr], y.iloc[itr])
        oof[iva] = m.predict(X.iloc[iva])
        maes.append(mean_absolute_error(y.iloc[iva], oof[iva]))
    lourds = y > q90                                        # les 10 % de sinistres les plus lourds
    lignes.append({
        "candidat": nom,
        "mae_cv": round(np.mean(maes), 2),
        "ecart_type": round(np.std(maes), 2),
        "mae_10pct_lourds": round(mean_absolute_error(y[lourds], oof[lourds]), 2),
        "biais_moyen": round((oof - y).mean(), 2),          # > 0 : on surestime en moyenne
        "temps_s": round(time.time() - t0, 1),
    })
    print(nom, "->", lignes[-1]["mae_cv"], f"({lignes[-1]['temps_s']} s)")

## Tableau de comparaison et gain sur le plancher

In [ ]:
tableau = pd.DataFrame(lignes)
plancher_mae = tableau.loc[0, "mae_cv"]                     # la 1re ligne est le plancher
tableau["gain_vs_plancher_%"] = ((1 - tableau["mae_cv"] / plancher_mae) * 100).round(1)
tableau = tableau.sort_values("mae_cv")
tableau.to_csv(OUT / "p3_2-6_comparaisons.csv", index=False)
tableau

## Graphique

In [ ]:
import matplotlib.pyplot as plt   # nécessite : pip install matplotlib

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(tableau["candidat"], tableau["mae_cv"], xerr=tableau["ecart_type"])
ax.axvline(plancher_mae, linestyle="--", label="plancher")
ax.set_xlabel("MAE en validation croisée (plus bas = mieux)")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "p3_2-6_comparaisons.png", dpi=120)
plt.show()

## Rapport

In [ ]:
meilleur = tableau.iloc[0]
R = ["# 2.6 Comparaisons",
     f"- plancher (médiane) : MAE {plancher_mae:,}",
     f"- meilleur candidat : {meilleur['candidat']} -> MAE {meilleur['mae_cv']:,} (gain {meilleur['gain_vs_plancher_%']} % sur le plancher)"]
for _, r in tableau.iterrows():
    R.append(f"- {r['candidat']} : MAE {r['mae_cv']:,} (± {r['ecart_type']:,}), 10 % lourds {r['mae_10pct_lourds']:,}, gain {r['gain_vs_plancher_%']} %, {r['temps_s']} s")
(OUT / "p3_2-6_comparaisons.md").write_text("\n".join(R), encoding="utf-8")
print("\n".join(R))

## Conclusion 2.6 *(à rédiger avec tes mots)*

> Le meilleur candidat est ..., avec une MAE de ... contre ... pour le plancher, soit un gain de ... %. Le gain est-il plus faible sur les 10 % de sinistres les plus lourds ? Le log a-t-il aidé ou nui, et pourquoi (apprendre la médiane de log(loss) n'est pas apprendre la médiane de loss, ni la moyenne) ? Quel temps de calcul, compatible avec les 10 minutes ?